In [1]:
import torch
from torch_geometric.nn import GraphConv, to_hetero
import torch 

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        self.conv = GraphConv(-1, hidden_channels)

    def forward(self, x, edge_index):

        x = self.conv(x, edge_index)
        return x

model = GCN( hidden_channels=5 )

In [3]:
from torch_geometric.data import HeteroData
from torch_geometric.utils import to_networkx


r"""
User(Bob) 0          User(Tom) 1 
          \         /       ↖
           \       /         \ 
            \     /           \
             ➘   ↙             ➘   
              Image <--------User (Dan) 2 
                0 
        (3 in homo graph)
"""

data                = HeteroData()
num_users           = 3
num_user_features   = 2
num_posts           = 1

data['user'].x  = torch.randn( num_users, num_user_features, dtype=torch.float )
data['image'].x = torch.empty( (1, 0),  dtype=torch.float )

data['user', 'follows', 'user'].edge_index = torch.tensor( [ [1], 
                                                             [2] ], dtype=torch.long )

data['user', 'post', 'image'].edge_index   = torch.tensor( [ [0], 
                                                             [0] ], dtype=torch.long )

data['user', 'likes', 'image'].edge_index  = torch.tensor( [ [1,2], 
                                                             [0,0] ], dtype=torch.long )

assert data.validate() is True
assert data.has_isolated_nodes() is False
assert data.has_self_loops() is False

homogenous_data = data.to_homogeneous()
graph = to_networkx(homogenous_data)
print(f"Graph edges are {graph.edges}") # 3 is the image 


Graph edges are [(0, 3), (1, 2), (1, 3), (2, 3)]


In [3]:
model = to_hetero(model, data.metadata())
print(f"Model is \n{model}")

out = model(data.x_dict, data.edge_index_dict)
print(f"Output is \n{out}")

print(f"\n\nWeights of the MODEL")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model is 
GraphModule(
  (conv): ModuleDict(
    (user__follows__user): GraphConv(-1, 5)
    (user__post__image): GraphConv(-1, 5)
    (user__likes__image): GraphConv(-1, 5)
  )
)



def forward(self, x, edge_index):
    x_dict = torch_geometric_nn_to_hetero_transformer_get_dict(x);  x = None
    x__user = x_dict.get('user', None)
    x__image = x_dict.get('image', None);  x_dict = None
    edge_index_dict = torch_geometric_nn_to_hetero_transformer_get_dict(edge_index);  edge_index = None
    edge_index__user__follows__user = edge_index_dict.get(('user', 'follows', 'user'), None)
    edge_index__user__post__image = edge_index_dict.get(('user', 'post', 'image'), None)
    edge_index__user__likes__image = edge_index_dict.get(('user', 'likes', 'image'), None);  edge_index_dict = None
    conv__user = self.conv.user__follows__user(x__user, edge_index__user__follows__user);  edge_index__user__follows__user = None
    conv__image1 = self.conv.user__post__image((x__user, x__image), edge_index

In [16]:
from torch_geometric.nn import HeteroConv, PowerMeanAggregation

class HeteroGNN(torch.nn.Module): 
    def __init__(self, hidden_channels):
        super(HeteroGNN, self).__init__()
        self.hidden_channels = hidden_channels  
        self.power_mean = PowerMeanAggregation(p=1.0, learn=True, channels=hidden_channels)
        aggr = ["sum"]
        self.conv1 = HeteroConv({
            ('user', 'follows', 'user'):    GraphConv(-1, hidden_channels, aggr = aggr),
            ('user', 'post', 'image'):      GraphConv(-1, hidden_channels, aggr = aggr),
            ('user', 'likes', 'image'):     GraphConv(-1, hidden_channels, aggr = aggr),
        }, aggr="cat")

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        print(f"After Conv shape is {x_dict['user'].shape}, {x_dict['image'].shape}")
        return x_dict
    
model = HeteroGNN(hidden_channels=5)
print(model)
output = model(data.x_dict, data.edge_index_dict)
print(f"\n\nUser {output['user'].shape} | Image {output['image'].shape}")

HeteroGNN(
  (power_mean): PowerMeanAggregation(learn=True)
  (conv1): HeteroConv(num_relations=3)
)
After Conv shape is torch.Size([3, 5]), torch.Size([1, 10])


User torch.Size([3, 5]) | Image torch.Size([1, 10])


In [101]:
num_nodes       = 5
incoming_types  = 3
output_channels = 4

input = torch.randn(num_nodes, incoming_types, output_channels)
print(f"Input is {input.shape}")
input = input.view(num_nodes * incoming_types, output_channels)
print(f"Input reshaped is {input.shape}")

index = torch.arange(incoming_types).repeat_interleave(num_nodes)
print(f"Index is {index}, with shape {index.shape}")

power_mean = PowerMeanAggregation(p=1.0, learn=True, channels=output_channels)
output = power_mean(input, index)

print(f"\nOutput is {output.shape},\n{output}")


Input is torch.Size([5, 3, 4])
Input reshaped is torch.Size([15, 4])
Index is tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]), with shape torch.Size([15])

Output is torch.Size([3, 4]),
tensor([[0.3324, 0.7316, 0.3480, 0.9962],
        [0.4709, 0.3610, 0.4245, 0.8073],
        [0.5580, 0.4791, 0.5209, 0.2184]], grad_fn=<PowBackward1>)


In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define dimensions
num_nodes = 10
node_types = 5
output_channels = 16

# Example input tensor of shape [num_nodes, node_types, output_channels]

# Define the Attention-Based Pooling
class AttentionPooling(nn.Module):
    def __init__(self, node_types, output_channels, hidden_dim):
        super(AttentionPooling, self).__init__()
        
        # Linear transformation for attention computation
        self.attention_layer = nn.Sequential(
            nn.Linear(output_channels, hidden_dim),  # Transform output_channels to hidden_dim
            nn.ReLU(),  # Non-linearity
            nn.Linear(hidden_dim, 1)  # Scalar attention score for each node_type
        )
        
        # Optional softmax for normalization (across node_types)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # x is of shape [num_nodes, node_types, output_channels]
        
        # Compute attention scores
        attention_scores = self.attention_layer(x)  # Shape: [num_nodes, node_types, 1]
        attention_scores = self.softmax(attention_scores)  # Normalize scores: [num_nodes, node_types, 1]
        
        # Multiply input by attention scores
        weighted_features = x * attention_scores  # Shape: [num_nodes, node_types, output_channels]
        
        # Aggregate across the node_types dimension
        aggregated_features = torch.sum(weighted_features, dim=1)  # Shape: [num_nodes, output_channels]
        
        return aggregated_features

# Instantiate the pooling layer
hidden_dim = 32  # Dimension of the hidden layer in the attention mechanism
attention_pooling = AttentionPooling(node_types=node_types, output_channels=output_channels, hidden_dim=hidden_dim)
input_tensor = torch.randn(13, node_types, output_channels)
print(f"Input tensor size is {input_tensor.shape}")

# Apply the pooling
pooled_output = attention_pooling(input_tensor)  # Shape: [num_nodes, output_channels]

# Print the result
print("Pooled output shape:", pooled_output.shape)

Input tensor size is torch.Size([13, 5, 16])
Pooled output shape: torch.Size([13, 16])


In [4]:
import torch 

embedding = torch.nn.Embedding(9, 3)
print(embedding.weight)

Parameter containing:
tensor([[-0.2787, -0.7047,  0.5498],
        [-0.2996, -1.7889, -0.7647],
        [ 0.1669,  1.1648, -0.2683],
        [-0.8465, -1.3259,  0.3352],
        [ 0.5395, -1.2983, -0.1247],
        [ 0.1322,  0.7397,  1.7718],
        [-0.6563,  0.5446,  1.4554],
        [-1.8980, -0.7788, -0.5713],
        [ 1.0889, -0.4354,  1.1357]], requires_grad=True)
